# Exercício 1 — Algoritmos Genéticos: conceitos, exploração de exemplos e melhoria com IA

Este notebook aborda os conceitos de evolução biológica, panoramas de adaptabilidade
(fitness landscapes) e algoritmos genéticos, seguindo as instruções da ementa da Aula 07.


## 1. Contexto

Nesta aula tratamos da contribuição da biologia para os estudos dos sistemas dinâmicos
de muitos agentes. Iniciamos com o desdobramento histórico dos modelos fenomenológicos
e matemáticos dos mecanismos genéticos.

- **Charles Darwin (1859)**: Teoria da evolução por seleção natural
- **Gregor Mendel (1865)**: Leis da hereditariedade
- **Síntese Moderna (1920–1940)**: Unificação da genética mendeliana com a seleção natural darwiniana
- **Sewall Wright (década de 1930)**: Conceito de **panorama de adaptação (fitness landscape)**,
  modelando a evolução genética de uma espécie em um dado ambiente
- **John Holland (década de 1960–1970)**: **Algoritmos genéticos** como ferramentas de
  otimização computacional baseadas nos princípios da evolução

Os algoritmos genéticos são apresentados em vídeos criados por **Victor Pellegrini Mammana**,
disponíveis no YouTube. Um exemplo clássico é o robô catador de latas de **Melanie Mitchell**,
implementado em NetLogo.


## 2. Célula auxiliar — imports e path discovery

Execute a célula abaixo para configurar os imports e os diretórios de saída.


In [3]:
# Garantir que python-dotenv esteja instalado
import subprocess, sys
try:
    from dotenv import load_dotenv
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "python-dotenv"])
    from dotenv import load_dotenv
load_dotenv()

import numpy as np
import matplotlib.pyplot as plt
import random
import os
import shutil
import subprocess
import glob

_repo = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_repo, "setup.sh")):
        break
    _repo = os.path.dirname(_repo)
PASTA_MOODLE = os.path.join(_repo, "work", "tarefas_moodle")
os.makedirs(PASTA_MOODLE, exist_ok=True)
PASTA_AULA = os.path.join(PASTA_MOODLE, "tarefa_aula07")
os.makedirs(PASTA_AULA, exist_ok=True)
os.environ["PASTA_AULA_SAVE"] = PASTA_AULA
print("Diretorio de saida:", PASTA_AULA)


error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

CalledProcessError: Command '['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'python-dotenv']' returned non-zero exit status 1.

## 3. Os 6 programas Python de exemplo

Os arquivos abaixo foram criados com o ChatGPT e estão disponíveis em `tarefas/aula07/`.
Eles exemplificam o uso de algoritmos genéticos para busca de máximos de funções.

| Arquivo | Função de fitness | Biblioteca | Visualização |
|---|---|---|---|
| `demo_alg_gen_GPT.py` | f(x) = x² | Custom (torneio) | 2D interativo (botão) |
| `demo_alg_gen_GPT_100-x2_func.py` | f(x) = 100 − x² | Custom (torneio) | 2D interativo (botão) |
| `demo_alg_gen_GPT_2MAX_Gauss_corrigido.py` | Duas Gaussianas (picos em 2 e 5) | Custom (roleta) | 2D interativo (botão) |
| `PanoramaExemplo.py` | sen(x)·cos(y) + 1.5·exp(−(x²+y²)/10) | Nenhum (só plot) | Superfície 3D estática |
| `Fitness.py` | sen(x)·cos(y) + 1.5·exp(−(x²+y²)/10) | DEAP | Contour 2D (a cada geração) |
| `Fitness_3D.py` | sen(x)·cos(y) + 1.5·exp(−(x²+y²)/10) | DEAP | Superfície 3D (geração final) |

**Instrução da ementa:** abra o Terminal e execute os programas:

```bash
cd tarefas/aula07
python3 demo_alg_gen_GPT.py
python3 demo_alg_gen_GPT_2MAX_Gauss_corrigido.py
python3 demo_alg_gen_GPT_100-x2_func.py
python3 PanoramaExemplo.py
python3 Fitness.py
python3 Fitness_3D.py
```

Os programas interativos abrirão janelas com botão "Próxima Geração".
Observe como o algoritmo genético converge para o máximo ao longo das gerações.


## 4. IA gera código de algoritmo genético

Vamos pedir para a IA gerar um algoritmo genético simples para maximizar
a função f(x) = 100 − x² no intervalo [−10, 10].

Execute a célula abaixo para que a IA gere o código.


In [ ]:
import os, json
from urllib.request import Request, urlopen

MODELO = os.getenv("OLLAMA_MODEL", "ministral-3:3b")
BASE_URL = os.getenv("OLLAMA_BASE_URL", "https://api.ollama.com").rstrip("/")
API_KEY = os.getenv("OLLAMA_API_KEY")

prompt_codigo = """Voce e um assistente de um curso de sistemas de muitos agentes.
Gere APENAS codigo Python (sem explicacoes) de um algoritmo genetico completo
para maximizar f(x) = 100 - x**2 no intervalo [-10, 10].
Use: populacao de 10 individuos, selecao por torneio (k=2), crossover blend,
mutacao gaussiana (sigma=1.0, taxa=0.2), 20 geracoes.
Ao final, gere um grafico com a curva de fitness ao longo das geracoes
e um scatter da populacao final sobre a funcao f(x).
O codigo deve ser AUTOSSUFICIENTE (com imports).
Nao use DEAP. Salve a figura em os.environ["PASTA_AULA_SAVE"].
"""

data = json.dumps({
    "model": MODELO,
    "prompt": prompt_codigo,
    "stream": False,
    "options": {"temperature": 0.2, "top_p": 0.9, "num_predict": 1500},
}).encode()

req = Request(BASE_URL + "/api/generate", data=data,
             headers={"Content-Type": "application/json"})
if API_KEY:
    req.add_header("Authorization", f"Bearer {API_KEY}")

try:
    resp = json.loads(urlopen(req, timeout=60).read().decode())
    codigo_gerado = resp["response"].strip()
    if "```python" in codigo_gerado:
        codigo_gerado = codigo_gerado.split("```python")[1]
        if "```" in codigo_gerado:
            codigo_gerado = codigo_gerado.split("```")[0]
    elif "```" in codigo_gerado:
        codigo_gerado = codigo_gerado.split("```")[1]
        if "```" in codigo_gerado:
            codigo_gerado = codigo_gerado.split("```")[0]
    codigo_gerado = codigo_gerado.strip()
    with open(os.path.join(PASTA_AULA, "codigo_ia_ag.py"), "w") as f:
        f.write(codigo_gerado)
    print("Codigo gerado com sucesso!")
    print(codigo_gerado[:500])
except Exception as e:
    print(f"Erro ao consultar IA: {type(e).__name__}: {e}")
    codigo_gerado = None


### 4.1. Executar o código gerado pela IA

Execute abaixo o código gerado pela IA. Verifique se o gráfico foi gerado corretamente.


In [ ]:
import os
os.environ["PASTA_AULA_SAVE"] = PASTA_AULA

codigo_path = os.path.join(PASTA_AULA, "codigo_ia_ag.py")
if os.path.exists(codigo_path):
    with open(codigo_path) as f:
        codigo = f.read()
    exec(codigo, {"__builtins__": __builtins__, "os": os, "np": np,
                   "plt": plt, "random": random, "PASTA_AULA": PASTA_AULA})
else:
    print("Codigo da IA nao encontrado. Execute a celula anterior primeiro.")


## 5. IA consulta: melhore o código

Vamos enviar um dos arquivos de exemplo para a IA analisar e sugerir melhorias,
conforme proposto na ementa.

O arquivo `PanoramaExemplo.py` apenas plota a superfície 3D da função de fitness.
Vamos pedir para a IA **adicionar um algoritmo genético** que encontre o máximo global
dessa função 2D e exiba o resultado.


In [ ]:
import os, json
from urllib.request import Request, urlopen

MODELO = os.getenv("OLLAMA_MODEL", "ministral-3:3b")
BASE_URL = os.getenv("OLLAMA_BASE_URL", "https://api.ollama.com").rstrip("/")
API_KEY = os.getenv("OLLAMA_API_KEY")

# Ler o arquivo PanoramaExemplo.py
with open(os.path.join(_repo, "tarefas", "aula07", "PanoramaExemplo.py")) as f:
    codigo_exemplo = f.read()

prompt_melhora = '''Voce e um assistente de um curso de sistemas de muitos agentes.
Analise o codigo Python abaixo e depois MELHORE-O adicionando um algoritmo genetico
completo (sem usar DEAP) para encontrar o maximo global da funcao de fitness 2D.
O AG deve usar: populacao de 20 individuos, crossover uniforme, mutacao gaussiana,
selecao por torneio, 30 geracoes.
Ao final, plote a superficie 3D com o melhor individuo destacado em vermelho.
O codigo melhorado deve ser AUTOSSUFICIENTE.
Salve a figura em os.environ["PASTA_AULA_SAVE"] como "aula_07_exercicio_01_melhorado.png".

CODIGO ORIGINAL:
```python
{codigo_exemplo}
```

Gere APENAS o codigo Python melhorado (sem explicacoes).
'''

prompt_melhora = prompt_melhora.format(codigo_exemplo=codigo_exemplo)

data = json.dumps({
    "model": MODELO,
    "prompt": prompt_melhora,
    "stream": False,
    "options": {"temperature": 0.2, "top_p": 0.9, "num_predict": 2000},
}).encode()

req = Request(BASE_URL + "/api/generate", data=data,
             headers={"Content-Type": "application/json"})
if API_KEY:
    req.add_header("Authorization", f"Bearer {API_KEY}")

try:
    resp = json.loads(urlopen(req, timeout=120).read().decode())
    codigo_melhorado = resp["response"].strip()
    if "```python" in codigo_melhorado:
        codigo_melhorado = codigo_melhorado.split("```python")[1]
        if "```" in codigo_melhorado:
            codigo_melhorado = codigo_melhorado.split("```")[0]
    elif "```" in codigo_melhorado:
        codigo_melhorado = codigo_melhorado.split("```")[1]
        if "```" in codigo_melhorado:
            codigo_melhorado = codigo_melhorado.split("```")[0]
    codigo_melhorado = codigo_melhorado.strip()
    with open(os.path.join(PASTA_AULA, "codigo_ia_melhorado.py"), "w") as f:
        f.write(codigo_melhorado)
    print("Codigo melhorado gerado com sucesso!")
    print(codigo_melhorado[:500])
except Exception as e:
    print(f"Erro ao consultar IA: {type(e).__name__}: {e}")
    codigo_melhorado = None


### 5.1. Executar o código melhorado pela IA

Execute abaixo o código melhorado pela IA.


In [ ]:
import os
os.environ["PASTA_AULA_SAVE"] = PASTA_AULA

codigo_path = os.path.join(PASTA_AULA, "codigo_ia_melhorado.py")
if os.path.exists(codigo_path):
    with open(codigo_path) as f:
        codigo = f.read()
    exec(codigo, {"__builtins__": __builtins__, "os": os, "np": np,
                   "plt": plt, "random": random, "PASTA_AULA": PASTA_AULA})
else:
    print("Codigo melhorado nao encontrado. Execute a celula anterior primeiro.")


## 6. Código curado (fallback)

Se a IA não gerou código válido, use a versão abaixo.
Implementa um GA completo para maximizar f(x) = 100 − x².


In [ ]:
# Versao alternativa do GA (fallback)
import os
import numpy as np
import matplotlib.pyplot as plt
import random

os.environ["PASTA_AULA_SAVE"] = PASTA_AULA

# Funcao de fitness
def fitness_function(x):
    return 100 - x ** 2

# Inicializar populacao
pop_size = 10
x_min, x_max = -10, 10
population = [random.uniform(x_min, x_max) for _ in range(pop_size)]

# Operadores geneticos
def selection(pop, fit_vals, k=2):
    return max(random.sample(list(zip(pop, fit_vals)), k), key=lambda x: x[1])[0]

def crossover(p1, p2, alpha=0.5):
    return p1 + alpha * (p2 - p1)

def mutate(x, sigma=1.0, rate=0.2):
    if random.random() < rate:
        x += random.gauss(0, sigma)
    return max(x_min, min(x_max, x))

# Evolucao
historico_melhor = []
historico_media = []

for gen in range(20):
    fitness_vals = [fitness_function(x) for x in population]
    historico_melhor.append(max(fitness_vals))
    historico_media.append(np.mean(fitness_vals))

    new_pop = []
    for _ in range(pop_size):
        p1 = selection(population, fitness_vals)
        p2 = selection(population, fitness_vals)
        child = crossover(p1, p2)
        child = mutate(child)
        new_pop.append(child)
    population = new_pop

# Plotar resultados
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Grafico de convergencia
ax1.plot(historico_melhor, "b-", label="Melhor fitness")
ax1.plot(historico_media, "r--", label="Fitness media")
ax1.set_xlabel("Geracao")
ax1.set_ylabel("Fitness")
ax1.set_title("Convergencia do Algoritmo Genetico")
ax1.legend()
ax1.grid(True)

# Populacao final sobre a funcao
xs = np.linspace(x_min, x_max, 200)
ax2.plot(xs, fitness_function(xs), "g-", label="f(x) = 100 - x^2")
final_fit = [fitness_function(x) for x in population]
ax2.scatter(population, final_fit, color="red", zorder=5, label="Populacao final")
melhor_idx = np.argmax(final_fit)
ax2.scatter([population[melhor_idx]], [final_fit[melhor_idx]], color="gold",
            s=200, marker="*", zorder=6, label="Melhor individuo")
ax2.set_xlabel("x")
ax2.set_ylabel("f(x)")
ax2.set_title("Populacao final sobre a funcao objetivo")
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(PASTA_AULA, "aula_07_exercicio_01_fallback.png"),
            dpi=150, bbox_inches="tight")
plt.show()
plt.close()
print(f"Melhor fitness encontrado: {max(final_fit):.4f} em x = {population[melhor_idx]:.4f}")
print("Figura salva como aula_07_exercicio_01_fallback.png")


## 7. Pergunta

> **Comente em poucas palavras: conseguiu rodar os programas? Consultou uma plataforma de IA
> para analisar e melhorar o código? O que mudou?**

*(máximo 100 palavras)*


In [ ]:
resposta_pergunta_1 = """
Escreva sua resposta aqui.
"""


## 8. Gerar resposta com IA

Execute a célula abaixo para que a IA analise seus resultados e gere uma resposta.


In [ ]:
import os, json
from urllib.request import Request, urlopen

MODELO = os.getenv("OLLAMA_MODEL", "ministral-3:3b")
BASE_URL = os.getenv("OLLAMA_BASE_URL", "https://api.ollama.com").rstrip("/")
API_KEY = os.getenv("OLLAMA_API_KEY")

prompt_resposta = """Voce e um assistente de um curso de sistemas de muitos agentes.
Responda em portugues, de forma clara e objetiva (maximo 100 palavras).

Pergunta: Comente em poucas palavras: conseguiu rodar os programas?
Consultou uma plataforma de IA para analisar e melhorar o codigo? O que mudou?

Contexto:
- Foram fornecidos 6 programas Python de exemplo sobre algoritmos geneticos.
- Os programas incluem versoes custom e usando a biblioteca DEAP.
- A IA foi consultada para gerar e melhorar codigos de GA.
"""

data = json.dumps({
    "model": MODELO,
    "prompt": prompt_resposta,
    "stream": False,
    "options": {"temperature": 0.3, "top_p": 0.9, "num_predict": 500},
}).encode()

req = Request(BASE_URL + "/api/generate", data=data,
             headers={"Content-Type": "application/json"})
if API_KEY:
    req.add_header("Authorization", f"Bearer {API_KEY}")

try:
    resp = json.loads(urlopen(req, timeout=60).read().decode())
    resposta = resp["response"].strip()
    print("Resposta da IA:")
    print(resposta)
except Exception as e:
    print(f"Erro ao consultar IA: {type(e).__name__}: {e}")
    print()
    print("Resposta atual:")
    print(resposta_pergunta_1)


## 9. Exportar para o Moodle

Execute a célula abaixo para gerar o arquivo Markdown e o PDF.


In [ ]:
import os
import shutil
import subprocess
from datetime import datetime

NOME = os.getenv("NOME", "nao informado")
RA = os.getenv("RA", "nao informado")
MODELO = os.getenv("OLLAMA_MODEL", "nao informado")

agora = datetime.now().strftime("%Y-%m-%d %H:%M")

PATH_MD = os.path.join(PASTA_AULA, "aula_07_exercicio_01.md")
with open(PATH_MD, "w", encoding="utf-8") as f:
    f.write("# Aula 07 — Algoritmos Geneticos: conceitos, exemplos e melhoria com IA\n\n")
    f.write(f"_Aluno: {NOME} (RA: {RA})_\n\n")
    f.write(f"_Modelo LLM: {MODELO}_\n\n")
    f.write("---\n\n")
    f.write(f"_Executado em: {agora}_\n\n")
    f.write("---\n\n")
    f.write("## Resposta do aluno\n\n")
    f.write(resposta_pergunta_1.strip())
    f.write("\n")

PATH_PDF = PATH_MD.replace(".md", ".pdf")
if shutil.which("pandoc"):
    subprocess.run(["pandoc", PATH_MD, "-o", PATH_PDF,
                   "--pdf-engine=xelatex", "--resource-path", PASTA_AULA,
                   "-V", "mainfont=DejaVu Serif"], check=True)
else:
    print("Aviso: pandoc nao instalado. Execute ./setup.sh para instalar.")
print(f"Markdown: {os.path.abspath(PATH_MD)}")
print(f"PDF:      {os.path.abspath(PATH_PDF)}")
